# Understanding remarkable_mouse/common.py

This notebook explains how the `remarkable_mouse` project connects to a reMarkable tablet via SSH and reads input events from `/dev/input/` files.


## Overview

The code connects to a reMarkable tablet via SSH (using Paramiko) and reads binary input events from Linux device files. These files follow the **evdev** (event device) protocol, which is the standard Linux kernel interface for input devices.


## 1. The evdev Input Event Format

Linux input events are stored in a binary format defined by the `input_event` structure in C:

```c
struct input_event {
    struct timeval time;  // Timestamp (seconds + microseconds)
    __u16 type;           // Event type (e.g., EV_ABS, EV_KEY, EV_SYN)
    __u16 code;           // Event code (e.g., ABS_X, ABS_Y, BTN_TOUCH)
    __s32 value;          // Event value (coordinate, pressure, button state, etc.)
};
```

The `struct timeval` contains:
- `tv_sec`: seconds (32-bit unsigned int)
- `tv_usec`: microseconds (32-bit unsigned int)

So the total structure size is:
- 8 bytes (timeval: 4 + 4)
- 2 bytes (type: __u16)
- 2 bytes (code: __u16)  
- 4 bytes (value: __s32)
- **Total: 16 bytes**

However, the code uses different format strings for different reMarkable models!


In [ ]:
import struct

# reMarkable 1 & 2 format
e_format_rm1 = '2IHHi'
# Breakdown:
# '2I' = 2 unsigned ints (32-bit each) = 8 bytes (timeval: tv_sec + tv_usec)
# 'H'  = 1 unsigned short (16-bit) = 2 bytes (type)
# 'H'  = 1 unsigned short (16-bit) = 2 bytes (code)
# 'i'  = 1 signed int (32-bit) = 4 bytes (value)
# Total: 8 + 2 + 2 + 4 = 16 bytes

# reMarkable Pro format
e_format_rmpro = 'I4xI4xHHi'
# Breakdown:
# 'I'   = 1 unsigned int (32-bit) = 4 bytes (tv_sec)
# '4x'  = 4 bytes of padding (skip)
# 'I'   = 1 unsigned int (32-bit) = 4 bytes (tv_usec)
# '4x'  = 4 bytes of padding (skip)
# 'H'   = 1 unsigned short (16-bit) = 2 bytes (type)
# 'H'   = 1 unsigned short (16-bit) = 2 bytes (code)
# 'i'   = 1 signed int (32-bit) = 4 bytes (value)
# Total: 4 + 4 + 4 + 4 + 2 + 2 + 4 = 24 bytes (with padding)

print(f"reMarkable 1/2 event size: {struct.calcsize(e_format_rm1)} bytes")
print(f"reMarkable Pro event size: {struct.calcsize(e_format_rmpro)} bytes")


## 2. Reading Input Events via SSH

The code uses Paramiko (SSH library) to execute commands on the reMarkable tablet. Instead of directly opening `/dev/input/event*` files (which would require root access and special handling), it uses the `dd` command to read the files in fixed-size chunks:

```python
cmd = f'dd bs={self.e_sz} if={self.pen_file}'
return self.client.exec_command(cmd, bufsize=self.e_sz, timeout=0)[1]
```

**Explanation:**
- `dd bs={self.e_sz}`: Read in blocks of size `e_sz` (16 or 24 bytes)
- `if={self.pen_file}`: Input file (e.g., `/dev/input/event0`)
- `exec_command()` returns a tuple: `(stdin, stdout, stderr)`
- `[1]` gets the stdout stream, which is a file-like object
- `bufsize=self.e_sz`: Set buffer size to match event size for efficiency
- `timeout=0`: No timeout (blocking read)


In [ ]:
# Example: How to parse a single input event
# This simulates what happens when reading from the stream

# Simulated binary data (16 bytes for reMarkable 1/2)
# In reality, this comes from reading the file stream
example_event_bytes = b'\x00' * 16  # Placeholder

# Unpack using struct
e_format = '2IHHi'  # reMarkable 1/2 format
unpacked = struct.unpack(e_format, example_event_bytes)

# Result: (tv_sec, tv_usec, type, code, value)
print(f"Unpacked event: {unpacked}")
print(f"  Time: {unpacked[0]}.{unpacked[1]:06d} seconds")
print(f"  Type: {unpacked[2]} (EV_ABS=3, EV_KEY=1, EV_SYN=0)")
print(f"  Code: {unpacked[3]} (e.g., ABS_X=0, ABS_Y=1)")
print(f"  Value: {unpacked[4]}")


## 3. Input Device Files

Different reMarkable models have different device file mappings:

**reMarkable 1:**
- Pen: `/dev/input/event0`
- Touch: `/dev/input/event2`
- Buttons: `/dev/input/event1`

**reMarkable 2:**
- Pen: `/dev/input/event1`
- Touch: `/dev/input/event2`
- Buttons: `/dev/input/event0`

**reMarkable Pro:**
- Pen: `/dev/input/event2`
- Touch: `/dev/input/event3`
- Buttons: `/dev/input/event0`

These are character device files that provide a stream of input events when read.


## 4. Event Types and Codes

The `type` field indicates what kind of event it is:
- `EV_SYN` (0): Synchronization event (marks end of a multi-event sequence)
- `EV_KEY` (1): Key/button event (press/release)
- `EV_ABS` (3): Absolute position event (X, Y, pressure, tilt, etc.)
- `EV_REL` (2): Relative movement event

The `code` field depends on the `type`:
- For `EV_ABS`: `ABS_X` (0), `ABS_Y` (1), `ABS_PRESSURE` (24), `ABS_TILT_X` (26), etc.
- For `EV_KEY`: `BTN_TOUCH` (330), `BTN_STYLUS` (331), etc.
- For `EV_SYN`: `SYN_REPORT` (0) - indicates all events in a frame have been sent

The `value` field depends on the code:
- For `ABS_X`/`ABS_Y`: Coordinate value (0 to max_x/max_y)
- For `ABS_PRESSURE`: Pressure value (0 to max_pressure)
- For `BTN_TOUCH`: 1 = pressed, 0 = released


In [ ]:
# Example: Understanding coordinate ranges

# reMarkable 1 pen coordinates
pen_x = (0, 20967, 100)  # (min, max, resolution)
pen_y = (0, 15725, 100)

print("reMarkable 1 Pen Coordinates:")
print(f"  X: {pen_x[0]} to {pen_x[1]} (resolution: {pen_x[2]})")
print(f"  Y: {pen_y[0]} to {pen_y[1]} (resolution: {pen_y[2]})")
print(f"  Physical size: ~{pen_x[1]/pen_x[2]:.1f}mm x {pen_y[1]/pen_y[2]:.1f}mm")

# When reading ABS_X events, the value will be between 0 and 20967
# This needs to be remapped to the target monitor's coordinates


## 5. Coordinate System Differences

The reMarkable tablets have different coordinate system orientations:

**reMarkable 1 & 2:**
- Pen: X increases rightward, Y increases downward (USB port at bottom)
- Touch: X increases leftward, Y increases downward (USB port at bottom)

**reMarkable Pro:**
- Both Pen and Touch: X increases rightward, Y increases downward (USB port at bottom)

The `remap()` function handles:
1. **Orientation transformation**: Rotates/flips coordinates based on where the USB port is
2. **Scaling**: Maps tablet coordinates to monitor coordinates using different modes:
   - `fill`: Scale to fill entire monitor (may crop)
   - `fit`: Scale to fit within monitor (may have letterboxing)
   - `stretch`: Stretch to fill monitor (may distort aspect ratio)


In [ ]:
# Example: How remapping works

def example_remap(x, y, max_x, max_y, monitor_width, monitor_height, mode='fit'):
    """Simplified remapping example"""
    # Calculate scaling ratios
    ratio_width = monitor_width / max_x
    ratio_height = monitor_height / max_y
    
    if mode == 'fill':
        # Use larger ratio to fill screen (may crop)
        scaling = max(ratio_width, ratio_height)
    elif mode == 'fit':
        # Use smaller ratio to fit screen (may letterbox)
        scaling = min(ratio_width, ratio_height)
    elif mode == 'stretch':
        # Different scaling for X and Y (distorts)
        scaling_x = ratio_width
        scaling_y = ratio_height
        return (scaling_x * x, scaling_y * y)
    
    # Center the scaled coordinates
    scaled_x = scaling * (x - (max_x - monitor_width / scaling) / 2)
    scaled_y = scaling * (y - (max_y - monitor_height / scaling) / 2)
    
    return (scaled_x, scaled_y)

# Example: reMarkable 1 pen at center
tablet_x, tablet_y = 20967 / 2, 15725 / 2
monitor_w, monitor_h = 1920, 1080

remapped = example_remap(tablet_x, tablet_y, 20967, 15725, monitor_w, monitor_h, 'fit')
print(f"Tablet center ({tablet_x}, {tablet_y}) -> Monitor: ({remapped[0]:.1f}, {remapped[1]:.1f})")


## 6. Complete Flow

Here's how the entire system works:

1. **SSH Connection**: Establish SSH connection to reMarkable tablet using Paramiko
2. **Open Streams**: Use `dd` command via SSH to open input device files as streams
3. **Read Events**: Continuously read fixed-size chunks (16 or 24 bytes) from streams
4. **Parse Events**: Use `struct.unpack()` to parse binary data into:
   - Timestamp (seconds, microseconds)
   - Event type (EV_ABS, EV_KEY, EV_SYN)
   - Event code (ABS_X, ABS_Y, BTN_TOUCH, etc.)
   - Event value (coordinate, pressure, button state)
5. **Process Events**: 
   - Track pen position from ABS_X/ABS_Y events
   - Track pressure from ABS_PRESSURE events
   - Track button states from EV_KEY events
   - Wait for SYN_REPORT to know a complete frame is ready
6. **Remap Coordinates**: Transform tablet coordinates to monitor coordinates
7. **Send to OS**: Use OS APIs (like uinput on Linux) to inject mouse/pen events

The key insight is that `/dev/input/event*` files provide a **stream of binary events** that must be read in fixed-size chunks and parsed using the struct format.


In [ ]:
# Example: Simulating reading and parsing events
# This shows what the actual code would do

def simulate_read_events(stream, e_format, num_events=5):
    """Simulate reading events from the stream"""
    events = []
    e_sz = struct.calcsize(e_format)
    
    for i in range(num_events):
        # In real code: data = stream.read(e_sz)
        # For demo, we'll create sample data
        import time
        now = time.time()
        tv_sec = int(now)
        tv_usec = int((now - tv_sec) * 1000000)
        
        # Create sample events
        if i == 0:
            # ABS_X event
            event_data = struct.pack(e_format, tv_sec, tv_usec, 3, 0, 10000)  # type=EV_ABS, code=ABS_X
        elif i == 1:
            # ABS_Y event
            event_data = struct.pack(e_format, tv_sec, tv_usec, 3, 1, 7500)   # type=EV_ABS, code=ABS_Y
        elif i == 2:
            # ABS_PRESSURE event
            event_data = struct.pack(e_format, tv_sec, tv_usec, 3, 24, 2048)  # type=EV_ABS, code=ABS_PRESSURE
        else:
            # SYN_REPORT event
            event_data = struct.pack(e_format, tv_sec, tv_usec, 0, 0, 0)      # type=EV_SYN, code=SYN_REPORT
        
        # Parse the event
        unpacked = struct.unpack(e_format, event_data)
        events.append({
            'time': (unpacked[0], unpacked[1]),
            'type': unpacked[2],
            'code': unpacked[3],
            'value': unpacked[4]
        })
    
    return events

# Simulate reading events
e_format = '2IHHi'
events = simulate_read_events(None, e_format, 5)

for i, event in enumerate(events):
    type_names = {0: 'EV_SYN', 1: 'EV_KEY', 3: 'EV_ABS'}
    code_names = {0: 'ABS_X', 1: 'ABS_Y', 24: 'ABS_PRESSURE'}
    
    print(f"Event {i+1}:")
    print(f"  Time: {event['time'][0]}.{event['time'][1]:06d}")
    print(f"  Type: {event['type']} ({type_names.get(event['type'], 'UNKNOWN')})")
    print(f"  Code: {event['code']} ({code_names.get(event['code'], 'UNKNOWN')})")
    print(f"  Value: {event['value']}")
    print()


# Connect to RM2

In [ ]:
import paramiko
import struct
import getpass
import sys
from paramiko.buffered_pipe import PipeTimeout

# reMarkable 2 configuration
RM2_IP = '192.168.101.13'
RM2_USER = 'root'  # Default user for reMarkable
RM2_PEN_FILE = '/dev/input/event1'  # Pen input for reMarkable 2
E_FORMAT = '2IHHi'  # Event format for reMarkable 1/2
E_SZ = struct.calcsize(E_FORMAT)

# Event type constants
EV_SYN = 0
EV_KEY = 1
EV_ABS = 3

# Event codes
ABS_X = 0
ABS_Y = 1
ABS_PRESSURE = 24
ABS_DISTANCE = 25
ABS_TILT_Y = 27
ABS_TILT_X = 26
BTN_TOOL_PEN = 320
BTN_TOUCH = 330
BTN_STYLUS = 331
SYN_REPORT = 0

# Type and code names for display
type_names = {
    EV_SYN: 'EV_SYN',
    EV_KEY: 'EV_KEY',
    EV_ABS: 'EV_ABS'
}

code_names = {
    EV_SYN: {
      SYN_REPORT: 'SYN_REPORT',
    },
    EV_KEY: {
      BTN_TOUCH: 'BTN_TOUCH',
      BTN_STYLUS: 'BTN_STYLUS',
      BTN_TOOL_PEN: 'BTN_TOOL_PEN',
    },
    EV_ABS: {
      ABS_X: 'ABS_X',
      ABS_Y: 'ABS_Y',
      ABS_PRESSURE: 'ABS_PRESSURE',
      ABS_DISTANCE: 'ABS_DISTANCE',
      ABS_TILT_Y: 'ABS_TILT_Y',
      ABS_TILT_X: 'ABS_TILT_X',
    }
}

print(f"Connecting to reMarkable 2 at {RM2_IP}...")

# Create SSH client
client = paramiko.SSHClient()
client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

# Get password from user
# password = getpass.getpass(f"Enter password for {RM2_USER}@{RM2_IP}: ")
password = 'CviBeVqbSg'

try:
    # Connect to reMarkable
    client.connect(RM2_IP, username=RM2_USER, password=password, timeout=10)
    print("✓ Connected successfully!")
    
    # Open pen input stream using dd command
    print(f"Opening pen input stream from {RM2_PEN_FILE}...")
    cmd = f'dd bs={E_SZ} if={RM2_PEN_FILE} 2>/dev/null'
    stdin, stdout, stderr = client.exec_command(cmd, bufsize=E_SZ, timeout=None)
    
    print("✓ Listening to pen events (press Ctrl+C to stop)...\n")
    print("Event format: [Time] Type Code Value")
    print("-" * 60)
    
    # Track current pen state
    pen_x = None
    pen_y = None
    pen_pressure = None
    pen_touching = False
    
    # Read events in a loop
    event_count = 0
    while True:
        try:
            # Read one event (E_SZ bytes)
            data = stdout.read(E_SZ)
            if len(data) < E_SZ:
                print("Stream closed or incomplete data")
                break
            
            # Parse the event
            tv_sec, tv_usec, e_type, e_code, e_value = struct.unpack(E_FORMAT, data)
            
            # Update pen state
            if e_type == EV_ABS:
                if e_code == ABS_X:
                    pen_x = e_value
                elif e_code == ABS_Y:
                    pen_y = e_value
                elif e_code == ABS_PRESSURE:
                    pen_pressure = e_value
            elif e_type == EV_KEY:
                if e_code == BTN_TOUCH:
                    pen_touching = (e_value == 1)
            
            # Display event
            event_count += 1
            type_str = type_names.get(e_type, f'UNKNOWN({e_type})')
            code_str = code_names.get(e_type, {}).get(e_code, f'UNKNOWN({e_code})')
            
            print(f"[{tv_sec}.{tv_usec:06d}] {type_str:10} {code_str:15} {e_value:8}")
            
            # When we get a SYN_REPORT, print current pen state summary
            if e_type == EV_SYN and e_code == SYN_REPORT:
                if pen_x is not None and pen_y is not None:
                    touch_status = "TOUCHING" if pen_touching else "HOVERING"
                    pressure_str = f", Pressure: {pen_pressure}" if pen_pressure is not None else ""
                    print(f"  → Pen at ({pen_x}, {pen_y}){pressure_str} - {touch_status}")
                    print()

        except PipeTimeout as e:
            print(f"\n\nTimeout: {e}")
            continue
        except KeyboardInterrupt as e:
            print(f"\n\nStopping event listener: {e}")
            break
        except Exception as e:
            print(f"\nError reading event: {e}")
            break
    
    print(f"\nTotal events received: {event_count}")
    
except paramiko.AuthenticationException:
    print("✗ Authentication failed. Please check your password.")
    sys.exit(1)
except paramiko.SSHException as e:
    print(f"✗ SSH connection error: {e}")
    sys.exit(1)
except Exception as e:
    print(f"✗ Error: {e}")
    sys.exit(1)
finally:
    client.close()
    print("Connection closed.")


[1763201704.084866] EV_ABS     ABS_Y               7177
[1763201704.084866] EV_SYN     SYN_REPORT             0
  → Pen at (5361, 7177), Pressure: 0 - HOVERING

[1763201704.086713] EV_ABS     ABS_Y               7176
[1763201704.086713] EV_ABS     ABS_DISTANCE          31
[1763201704.086713] EV_SYN     SYN_REPORT             0
  → Pen at (5361, 7176), Pressure: 0 - HOVERING

[1763201704.090408] EV_ABS     ABS_Y               7175
[1763201704.090408] EV_ABS     ABS_DISTANCE          32
[1763201704.090408] EV_SYN     SYN_REPORT             0
  → Pen at (5361, 7175), Pressure: 0 - HOVERING

[1763201704.095964] EV_ABS     ABS_Y               7174
[1763201704.095964] EV_SYN     SYN_REPORT             0
  → Pen at (5361, 7174), Pressure: 0 - HOVERING

[1763201704.097802] EV_ABS     ABS_X               5362
[1763201704.097802] EV_SYN     SYN_REPORT             0
  → Pen at (5362, 7174), Pressure: 0 - HOVERING

[1763201704.099649] EV_ABS     ABS_Y               7173
[1763201704.099649] EV_ABS 